# Final-Test Visualizations: Coefficient of Determination and RMSE Maps

Visualizes the final-test results (2023–2024, 93 grids × 24 months = 2,232 forecasts per model) of the **three standalone models** (SARIMAX, LSTM, XGBoost) and the **three stacked hybrid models** (Hybrid SARIMAX-LSTM, + CSI, + CSI + PL).

| Figure | What it shows |
|---|---|
| **Coefficient of determination graph** | Forecast vs observed GHI for every test forecast, one panel per model, with the 1:1 line. Each panel states the pooled R² and pooled RMSE. |
| **RMSE maps** | Pooled RMSE of each grid cell (over its 24 test months) on the Philippine spatial grid. All six maps share **one colour scale**, so a colour means the same error in every panel. |

All metrics are pooled, as in `final_test_metrics.csv`:
- R² = 1 − Σ(ŷ − y)² / Σ(y − ȳ)², over all 2,232 forecasts
- per-grid RMSE = √(mean of that grid's 24 squared errors)

**Outputs** (`outputs/figures/`, each as a 300-dpi PNG, an SVG and a PDF):
- `final_test_r2_scatter`: coefficient of determination graph (6 panels)
- `final_test_rmse_maps`: RMSE maps (6 panels, shared scale)
- `rmse_maps/<model>`: one RMSE map per model, same scale, if you need them separately
- `final_test_rmse_by_grid.csv`: the values behind the maps (grid × model), the table equivalent of the maps

Run after `Final_Testing_Standalone.ipynb` and `Final_Testing_Hybrid.ipynb`, from the `Philippine Spatial Map` folder.

In [ ]:
"""
final_test_visualizations.py

Purpose:
    Coefficient of determination graph and per-grid pooled RMSE maps
    for the final test (2023-2024).

Input:
    outputs/final_test/predictions/
    outputs/final_test_hybrid/predictions/
    data/processed/philippines_grid.geojson
    data/boundaries/Philippines.shp/PH_Adm0_Country.shp.shp

Output:
    outputs/figures/
"""

from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable


# ============================================================
# 1. USER SETTINGS
# ============================================================

STANDALONE_FINAL = Path("outputs/final_test/predictions")

HYBRID_FINAL = Path("outputs/final_test_hybrid/predictions")

GRID_FILE = Path("data/processed/philippines_grid.geojson")

BOUNDARY_FILE = Path("data/boundaries/Philippines.shp/PH_Adm0_Country.shp.shp")

OUTPUT_DIR = Path("outputs/figures")

MAP_DIR = OUTPUT_DIR / "rmse_maps"


# Panel order: standalone models (top row), stacked hybrids (bottom row)
MODELS = {
    "SARIMAX":                           STANDALONE_FINAL / "sarimax_final_test_predictions.csv",
    "LSTM":                              STANDALONE_FINAL / "lstm_final_test_predictions.csv",
    "XGBoost":                           STANDALONE_FINAL / "xgboost_final_test_predictions.csv",
    "Hybrid SARIMAX-LSTM":               HYBRID_FINAL / "hybrid_final_test_predictions.csv",
    "Hybrid SARIMAX-LSTM + CSI":         HYBRID_FINAL / "hybrid_csi_final_test_predictions.csv",
    "Hybrid SARIMAX-LSTM + CSI + PL":    HYBRID_FINAL / "hybrid_csi_pl_final_test_predictions.csv"
}

PROPOSED = "Hybrid SARIMAX-LSTM + CSI + PL"

# Short panel titles (the row heading names the model family)
PANEL_TITLES = {
    "SARIMAX":                        "SARIMAX",
    "LSTM":                           "LSTM",
    "XGBoost":                        "XGBoost",
    "Hybrid SARIMAX-LSTM":            "Hybrid SARIMAX-LSTM",
    "Hybrid SARIMAX-LSTM + CSI":      "Hybrid + CSI",
    "Hybrid SARIMAX-LSTM + CSI + PL": "Hybrid + CSI + PL"
}

ROW_HEADINGS = ["Standalone models", "Stacked hybrid models"]

UNITS = "kWh/m²/day"


# ------------------------------------------------------------
# Style: one hue for points, one-hue blue ramp for magnitude
# ------------------------------------------------------------

INK = "#0b0b0b"          # primary text
INK_2 = "#52514e"        # secondary text
MUTED = "#898781"        # reference line, tick labels
GRIDLINE = "#e1e0d9"     # hairline grid
AXIS = "#c3c2b7"         # axis lines
POINT = "#2a78d6"        # single-series colour

# Sequential blue ramp (light = low error, dark = high error)
RAMP = ["#cde2fb", "#b7d3f6", "#9ec5f4", "#86b6ef", "#6da7ec", "#5598e7", "#3987e5",
        "#2a78d6", "#256abf", "#1c5cab", "#184f95", "#104281", "#0d366b"]

RMSE_CMAP = LinearSegmentedColormap.from_list("rmse_blue", RAMP)

plt.rcParams.update({
    "font.family":      "Arial",
    "mathtext.fontset": "custom",
    "mathtext.rm":      "Arial",
    "mathtext.it":      "Arial:italic",
    "mathtext.bf":      "Arial:bold",
    "font.size":        7.5,
    "axes.edgecolor":   AXIS,
    "axes.linewidth":   0.6,
    "axes.labelcolor":  INK_2,
    "xtick.color":      MUTED,
    "ytick.color":      MUTED,
    "xtick.labelcolor": INK_2,
    "ytick.labelcolor": INK_2,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 2.5,
    "ytick.major.size": 2.5,
    "svg.fonttype":     "none",
    "pdf.fonttype":     42
})

for directory in [OUTPUT_DIR, MAP_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


def add_axes_inches(fig, left, top, width, height):
    """Add axes positioned in inches from the figure's top-left corner."""

    fig_width, fig_height = fig.get_size_inches()

    return fig.add_axes([left / fig_width, (fig_height - top - height) / fig_height,
                         width / fig_width, height / fig_height])


def save(fig, path_stem):
    """Save PNG (300 dpi), SVG and PDF."""

    for extension in ("png", "svg", "pdf"):
        fig.savefig(f"{path_stem}.{extension}", dpi=300, bbox_inches="tight",
                    pad_inches=0.04, facecolor="white")

    print(f"Saved: {path_stem}.png / .svg / .pdf")


# ============================================================
# 2. LOAD FORECASTS AND COMPUTE POOLED METRICS
# ============================================================

forecasts = {}

for model, path in MODELS.items():

    if not path.exists():
        raise FileNotFoundError(f"{path} not found. Run the final-test notebooks first.")

    frame = pd.read_csv(path, parse_dates=["target_date"])

    if len(frame) != 2232 or frame[["actual", "predicted"]].isna().any().any():
        raise ValueError(f"{model}: expected 2,232 complete forecasts, found {len(frame)}")

    forecasts[model] = frame


def pooled(actual, predicted):
    """Pooled RMSE and R² (1 - SSE/SST) over all forecasts given."""

    error = predicted - actual

    sse = np.sum(error ** 2)

    sst = np.sum((actual - actual.mean()) ** 2)

    return np.sqrt(sse / len(actual)), 1 - sse / sst


overall = pd.DataFrame(
    [
        dict(zip(["RMSE", "R2"], pooled(frame["actual"], frame["predicted"])), model=model)
        for model, frame in forecasts.items()
    ]
).set_index("model")


# Per-grid pooled RMSE (24 test months per grid)
rmse_by_grid = pd.DataFrame({
    model: frame.assign(se=(frame["predicted"] - frame["actual"]) ** 2)
                .groupby("grid_number")["se"].mean() ** 0.5
    for model, frame in forecasts.items()
})

rmse_by_grid.index.name = "grid_number"

rmse_by_grid.to_csv(OUTPUT_DIR / "final_test_rmse_by_grid.csv")

print("Pooled final-test metrics (2023-2024, 2,232 forecasts per model):")

print(overall.round(4).to_string())

print("\nPer-grid RMSE range across all six models: "
      f"{rmse_by_grid.min().min():.3f} - {rmse_by_grid.max().max():.3f} {UNITS}")

## 1. Coefficient of determination graph

Each panel plots the forecast GHI against the observed GHI for all 2,232 test forecasts of one model. Points on the **1:1 line** are perfect forecasts, so the closer the cloud sits to the line, the higher the R². All panels use the same axes so they can be compared directly.

In [ ]:
# ============================================================
# 3. COEFFICIENT OF DETERMINATION GRAPH
# ============================================================

values = np.concatenate([np.r_[f["actual"], f["predicted"]] for f in forecasts.values()])

low = np.floor(values.min() * 2) / 2

high = np.ceil(values.max() * 2) / 2

ticks = np.arange(np.ceil(low), np.floor(high) + 1)

# Layout in inches: 3 square panels per row, row heading + title above each row
FIG_W, LEFT, PANEL, GUTTER = 6.5, 0.52, 1.84, 0.2
ROW_HEADING, TITLE_BAND, ROW_GAP, BOTTOM = 0.2, 0.22, 0.24, 0.42

FIG_H = 0.05 + 2 * (ROW_HEADING + TITLE_BAND + PANEL) + ROW_GAP + BOTTOM

fig = plt.figure(figsize=(FIG_W, FIG_H))

for index, (model, frame) in enumerate(forecasts.items()):

    row, column = divmod(index, 3)

    row_top = 0.05 + row * (ROW_HEADING + TITLE_BAND + PANEL + ROW_GAP)

    if column == 0:
        fig.text(0.02 / FIG_W, 1 - row_top / FIG_H, ROW_HEADINGS[row], fontsize=8,
                 fontweight="bold", color=INK_2, va="top", ha="left")

    ax = add_axes_inches(fig, LEFT + column * (PANEL + GUTTER),
                         row_top + ROW_HEADING + TITLE_BAND, PANEL, PANEL)

    ax.grid(color=GRIDLINE, linewidth=0.5, zorder=0)

    ax.plot([low, high], [low, high], color=MUTED, linewidth=0.9, zorder=1)

    ax.text(low + 0.55, low + 0.28, "1:1 line", ha="left", va="top", fontsize=6.5, color=INK_2)

    ax.scatter(frame["actual"], frame["predicted"], s=3.5, color=POINT, alpha=0.28,
               linewidths=0, zorder=2, rasterized=True)

    rmse, r2 = overall.loc[model, "RMSE"], overall.loc[model, "R2"]

    ax.text(low + 0.12, high - 0.12, f"R$^2$ = {r2:.3f}", ha="left", va="top",
            fontsize=8, fontweight="bold", color=INK)

    ax.text(low + 0.12, high - 0.55, f"RMSE = {rmse:.3f}", ha="left", va="top",
            fontsize=7, color=INK_2)

    if model == PROPOSED:
        ax.text(high - 0.1, low + 0.15, "Proposed model", ha="right", va="bottom",
                fontsize=6.8, style="italic", color=INK_2)

    ax.set_title(f"({'abcdef'[index]}) {PANEL_TITLES[model]}", fontsize=7.6,
                 fontweight="bold", color=INK, loc="left", pad=4)

    ax.set_xlim(low, high)
    ax.set_ylim(low, high)
    ax.set_xticks(ticks)
    ax.set_yticks(ticks)
    ax.tick_params(labelbottom=row == 1, labelleft=column == 0)

    if row == 1:
        ax.set_xlabel(f"Observed GHI ({UNITS})")

    if column == 0:
        ax.set_ylabel(f"Forecast GHI ({UNITS})")

    for side in ("top", "right"):
        ax.spines[side].set_visible(False)

save(fig, OUTPUT_DIR / "final_test_r2_scatter")

plt.show()

## 2. RMSE maps on the Philippine spatial grid

Each grid cell is coloured by its pooled RMSE over the 24 test months, **darker = larger error**. All six maps use the same colour scale, so differences in shade between panels are real differences in error. The values are in `final_test_rmse_by_grid.csv`.

In [ ]:
# ============================================================
# 4. RMSE MAPS
# ============================================================

grid = gpd.read_file(GRID_FILE).merge(rmse_by_grid.reset_index(), on="grid_number", how="left")

if grid[list(MODELS)].isna().any().any():
    raise ValueError("Some grid cells have no RMSE")

boundary = gpd.read_file(BOUNDARY_FILE).to_crs(grid.crs)

boundary["geometry"] = boundary.geometry.simplify(0.005)

# One colour scale for every map
VMIN = np.floor(rmse_by_grid.min().min() * 20) / 20

VMAX = np.ceil(rmse_by_grid.max().max() * 20) / 20

NORM = Normalize(vmin=VMIN, vmax=VMAX)

LON_LIMITS, LAT_LIMITS = (115.8, 127.2), (3.8, 22.2)

MAP_ASPECT = (LAT_LIMITS[1] - LAT_LIMITS[0]) / (LON_LIMITS[1] - LON_LIMITS[0])


def draw_map(ax, model, show_x=True, show_y=True):

    grid.plot(ax=ax, column=model, cmap=RMSE_CMAP, norm=NORM,
              edgecolor="white", linewidth=0.5, zorder=1)

    boundary.plot(ax=ax, facecolor="none", edgecolor=INK_2, linewidth=0.3, zorder=2)

    ax.set_xlim(*LON_LIMITS)
    ax.set_ylim(*LAT_LIMITS)
    ax.set_aspect("equal")
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticks([118, 122, 126])
    ax.set_yticks([6, 10, 14, 18, 22])
    ax.set_xticklabels([f"{t}°E" for t in [118, 122, 126]])
    ax.set_yticklabels([f"{t}°N" for t in [6, 10, 14, 18, 22]])
    ax.tick_params(labelsize=6.5, labelbottom=show_x, labelleft=show_y)

    for side in ax.spines.values():
        side.set_color(AXIS)
        side.set_linewidth(0.6)


def subtitle(model):

    text = (f"RMSE {overall.loc[model, 'RMSE']:.3f} "
            f"(range {rmse_by_grid[model].min():.2f}–{rmse_by_grid[model].max():.2f})")

    return ("Proposed · " + text) if model == PROPOSED else text


def add_colorbar(fig, cax, label):

    colorbar = fig.colorbar(ScalarMappable(norm=NORM, cmap=RMSE_CMAP), cax=cax)

    colorbar.set_label(label, fontsize=7, color=INK_2)

    colorbar.outline.set_visible(False)

    colorbar.ax.tick_params(labelsize=6.5, length=2)


# ------------------------------------------------------------
# Combined figure: 2 x 3 maps, one shared colour bar
# ------------------------------------------------------------

FIG_W, LEFT, PANEL_W, GUTTER = 6.5, 0.45, 1.7, 0.15
PANEL_H = PANEL_W * MAP_ASPECT
ROW_HEADING, TITLE_BAND, ROW_GAP, BOTTOM = 0.2, 0.34, 0.16, 0.24

FIG_H = 0.05 + 2 * (ROW_HEADING + TITLE_BAND + PANEL_H) + ROW_GAP + BOTTOM

fig = plt.figure(figsize=(FIG_W, FIG_H))

for index, model in enumerate(MODELS):

    row, column = divmod(index, 3)

    row_top = 0.05 + row * (ROW_HEADING + TITLE_BAND + PANEL_H + ROW_GAP)

    if column == 0:
        fig.text(0.02 / FIG_W, 1 - row_top / FIG_H, ROW_HEADINGS[row], fontsize=8,
                 fontweight="bold", color=INK_2, va="top", ha="left")

    ax = add_axes_inches(fig, LEFT + column * (PANEL_W + GUTTER),
                         row_top + ROW_HEADING + TITLE_BAND, PANEL_W, PANEL_H)

    draw_map(ax, model, show_x=row == 1, show_y=column == 0)

    ax.text(0.0, 1.0 + 0.2 / PANEL_H, f"({'abcdef'[index]}) {PANEL_TITLES[model]}",
            transform=ax.transAxes, fontsize=7.6, fontweight="bold", color=INK, ha="left", va="bottom")

    ax.text(0.0, 1.0 + 0.04 / PANEL_H, subtitle(model), transform=ax.transAxes,
            fontsize=6.6, color=INK_2, ha="left", va="bottom")

panels_top = 0.05 + ROW_HEADING + TITLE_BAND

panels_bottom = FIG_H - BOTTOM

colorbar_ax = add_axes_inches(fig, LEFT + 3 * PANEL_W + 2 * GUTTER + 0.12,
                              panels_top + 0.6, 0.1, panels_bottom - panels_top - 1.2)

add_colorbar(fig, colorbar_ax, f"Pooled RMSE per grid cell, 2023–2024 ({UNITS})")

save(fig, OUTPUT_DIR / "final_test_rmse_maps")

plt.show()


# ------------------------------------------------------------
# One map per model (same colour scale), for separate use
# ------------------------------------------------------------

for model in MODELS:

    panel_w = 2.2

    panel_h = panel_w * MAP_ASPECT

    fig = plt.figure(figsize=(3.5, 0.05 + 0.42 + panel_h + 0.25))

    ax = add_axes_inches(fig, 0.45, 0.47, panel_w, panel_h)

    draw_map(ax, model)

    ax.text(0.0, 1.0 + 0.2 / panel_h, model, transform=ax.transAxes,
            fontsize=7.8, fontweight="bold", color=INK, ha="left", va="bottom")

    ax.text(0.0, 1.0 + 0.04 / panel_h, subtitle(model), transform=ax.transAxes,
            fontsize=6.8, color=INK_2, ha="left", va="bottom")

    add_colorbar(fig, add_axes_inches(fig, 0.45 + panel_w + 0.12, 0.47 + 0.5, 0.1, panel_h - 1.0),
                 f"Pooled RMSE, 2023–2024 ({UNITS})")

    file_name = model.lower().replace(" + ", "_").replace(" ", "_").replace("-", "_")

    save(fig, MAP_DIR / file_name)

    plt.close(fig)